# Household-Level Tract Choice Demo with livelike, geosnap, and locpick

This notebook demonstrates a synthetic-population workflow for estimating a household-level tract choice model.

The pipeline is:

1. Build tract alternatives from ACS with `geosnap.get_acs(...)`.
2. Build a tract-targeted PUMA object with `livelike.acs.puma(...)`.
3. Solve the P-MEDM allocation problem with `pymedm.PMEDM`.
4. Generate one synthetic household placement draw with `livelike.homesim.synthesize(...)`.
5. Expand household counts into one chooser row per synthetic household instance.
6. Join household attributes from `livelike.attribution.build_attributes(...)`.
7. Estimate a tract choice model with `locpick.MNL`.

This is a synthetic allocation workflow, not a revealed-preference migration model. The chosen tract comes from the synthetic allocation output.


## Environment Notes

Run this notebook in the `locpick` conda environment. The examples below assume the environment has at least these packages installed:

- `livelike`
- `pymedm`
- `geosnap`
- `locpick`

If you have both a local `geosnap` checkout and an installed `geosnap`, run the notebook from the `choicemodels` repository so the installed package resolves cleanly.


In [1]:
%load_ext autoreload
%autoreload 2


import os

import numpy as np
import pandas as pd
from geosnap import DataStore
from geosnap.io import get_acs
from livelike import acs, homesim
from livelike.attribution import build_attributes
from livelike.config import up_expanded_attributes_household
from pymedm import PMEDM

from locpick import ChoiceModel, ChoiceTable

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## Study Settings

Set the study year, PUMA, and county filter for tract alternatives.

> This version of `livelike` strips leading zeros from the 5-digit PUMA code via `str(int(fips[2:]))`, so PUMAs with codes < 10000 (e.g. Alabama PUMA `00100`) generate broken Census API URLs. All six Ventura County PUMAs (11101–11106) have codes ≥ 10000 and are unaffected.
> **Why Ventura County, CA?**  


In [2]:
YEAR = 2019

# PUMA 0611101 - Ventura County (Southeast), Simi Valley, California.
# All Ventura County PUMAs (11101-11106) have 5-digit codes >= 10000,
# so livelike's int() conversion does not strip leading zeros.
STATE_FIPS = "06"  # California
PUMA_FIPS = "0611101"  # Ventura County (Southeast) - Simi Valley
COUNTY_FIPS = "06111"  # Ventura County

CENSUS_API_KEY = os.environ.get("CENSUS")

SIM_ID = 0
RANDOM_STATE = 42

## Build the Tract Alternatives Table

For the first pass, keep the tract alternatives compact and interpretable. The tract `geoid` becomes the `locpick` alternative id.


In [3]:
store = DataStore()

tracts = get_acs(
    datastore=store,
    county_fips=COUNTY_FIPS,
    level="tract",
    years=[YEAR],
)

alt_cols = [
    "median_home_value",
    "median_contract_rent",
    "median_household_income",
    "per_capita_income",
    "p_poverty_rate",
]

# Keep all tracts that appear in the PUMA and fill missing values with 0
alternatives = tracts.set_index("geoid").loc[:, alt_cols].copy()
alternatives = alternatives.fillna(0.0)

# Standardize alternative-specific variables for numerical stability
for col in alt_cols:
    mean = alternatives[col].mean()
    std = alternatives[col].std()
    if std > 0:
        alternatives[col] = (alternatives[col] - mean) / std

alternatives.index = alternatives.index.astype(str)
alternatives.index.name = "alt_id"

alternatives.head()

,median_home_value,median_contract_rent,median_household_income,per_capita_income,p_poverty_rate
alt_id,,,,,
06111001301,-0.211500,-0.124185,0.288306,-0.041713,0.032449
06111003011,-0.399385,-0.794209,-1.038224,-0.935906,0.333310
06111007201,1.364402,1.264213,2.105388,1.632452,-0.861669
06111007402,1.495962,0.915387,1.401131,2.863192,-1.000166
06111007507,0.868739,2.801123,2.162357,2.049955,-1.094218


## Build the livelike PUMA Object

Use `target_zone="trt"` so the allocation targets are census tracts.

The `keep_intermediates=True` flag is important here because `build_attributes(..., level="household", ...)` requires `puma.est_household` to be retained.


In [4]:
pup = acs.puma(
    fips=PUMA_FIPS,
    target_zone="trt",
    year=YEAR,
    keep_intermediates=True,
    keep_geo=True,
    censusapikey=CENSUS_API_KEY,
    random_state=RANDOM_STATE,
)

pup.est_ind.head()

,population,group_quarters_pop,housing_units,occhu,civ_noninst_pop,male_hours_GE35,male_hours_15.34,male_hours_1.14,female_hours_GE35,female_hours_15.34,...,hht_fam_hhsize_5p,hht_fam_hhsize_6p,hht_fam_hhsize_7pm,hht_nonfam_hhsize_1p,hht_nonfam_hhsize_2p,hht_nonfam_hhsize_3p,hht_nonfam_hhsize_4p,hht_nonfam_hhsize_5p,hht_nonfam_hhsize_6p,hht_nonfam_hhsize_7pm
SERIALNO,,,,,,,,,,,,,,,,,,,,,
2015000001402,2.285714,0.0,1,1,2.285714,0.0,0.0,0.0,0.000000,1.00,...,0,0,0,0,1,0,0,0,0,0
2015000009563,1.000000,1.0,0,0,0.000000,0.0,0.0,0.0,0.000000,0.00,...,0,0,0,0,0,0,0,0,0,0
2015000018798,4.750000,0.0,1,1,4.750000,1.0,0.0,0.0,1.333333,1.25,...,0,0,0,0,0,0,0,0,0,0
2015000018881,3.769231,0.0,1,1,3.769231,1.0,0.0,0.0,0.000000,0.00,...,0,0,0,0,0,0,0,0,0,0
2015000020883,1.000000,0.0,1,1,1.000000,0.0,0.0,0.0,0.000000,0.00,...,0,0,0,1,0,0,0,0,0,0


## Solve the P-MEDM Allocation

The allocation step uses the exact `PMEDM(...)` call shape exercised by `livelike`'s installed test data recipe.

The resulting `pmd.almat` matrix is the input to household placement synthesis.


In [5]:
pmd = PMEDM(
    pup.year,
    pup.est_ind.index,
    pup.wt,
    pup.est_ind,
    pup.est_g1,
    pup.est_g2,
    pup.se_g1,
    pup.se_g2,
    topo=pup.topo,
    random_state=RANDOM_STATE,
)

pmd.solve()

float(pmd.res.state.value)

-0.7022985122256593

## Synthesize Household Placements

`homesim.synthesize(..., longform=True)` returns a long household placement table indexed by `h_id` with `sim`, `geoid`, and `count` columns.

For the first estimation demo, use a single simulation draw.


In [6]:
hs = homesim.synthesize(
    almat=pmd.almat,
    est_ind=pup.est_ind,
    est_g2=pup.est_g2,
    sporder=pup.sporder,
    nsim=5,
    random_state=RANDOM_STATE,
    longform=True,
)

hs = hs.reset_index()
hs = hs.loc[hs["sim"] == SIM_ID].copy()
hs.head()

,h_id,sim,geoid,count
0,2015000024098,0,06111007505,1
1,2015000029822,0,06111007505,1
2,2015000040352,0,06111007505,1
3,2015000049735,0,06111007505,1
4,2015000052524,0,06111007505,1


## Expand Counts into One Chooser Row Per Synthetic Household Instance

The synthetic placement table counts how many times each household id is assigned to each tract in a given simulation. For `locpick`, expand those counts so there is one chooser row per synthetic household instance.


In [7]:
choosers_long = hs.loc[hs.index.repeat(hs["count"])].copy()
choosers_long["chooser_id"] = np.arange(len(choosers_long))
choosers_long = choosers_long.rename(columns={"geoid": "chosen_tract"})
choosers_long = choosers_long.drop(columns=["count", "sim"])
choosers_long = choosers_long.set_index("chooser_id")
choosers_long.index.name = "obs_id"

choosers_long.head()

,h_id,chosen_tract
obs_id,,
0,2015000024098,06111007505
1,2015000029822,06111007505
2,2015000040352,06111007505
3,2015000049735,06111007505
4,2015000052524,06111007505


## Attach Household Attributes

The expanded household attribute preset provides a compact household-level profile suitable for a first tract choice model.

The join below assumes the `h_id` domain returned by `homesim.synthesize(...)` aligns with the household index retained in `pup.est_household`. The assertion checks that assumption before estimation proceeds.


In [8]:
# Build household attributes from the livelike PUMA object.
# Note: not all attributes in up_expanded_attributes_household may be available
# for every ACS vintage / geography. We filter to those that succeed.
_available_atts = []
for _att in up_expanded_attributes_household:
    try:
        build_attributes(pup, level="household", variables=[_att])
        _available_atts.append(_att)
    except Exception:
        pass

household_atts = build_attributes(
    pup,
    level="household",
    variables=_available_atts,
)

if not pd.Index(choosers_long["h_id"]).isin(household_atts.index).all():
    raise ValueError(
        "Some synthesized household ids are missing from the household attribute table. "
        "Inspect the livelike household index before continuing."
    )

choosers = choosers_long.join(household_atts, on="h_id", how="left")
choosers["chosen_tract"] = choosers["chosen_tract"].astype(str)

# Keep only columns that actually exist in the joined frame
chooser_cols = ["h_id", "chosen_tract"] + [
    c for c in household_atts.columns if c in choosers.columns
]

choosers = choosers.loc[:, chooser_cols].copy()

## Prepare Interaction Variables

For a first model, include a small number of chooser-by-tract interactions that capture affordability and tenure-market fit.


In [9]:
obs_ids = choosers.index.to_numpy()
alt_ids = alternatives.index.to_numpy()
cross = pd.MultiIndex.from_product([obs_ids, alt_ids], names=["obs_id", "alt_id"])
interaction_frame = pd.DataFrame(index=cross)

obs_index = interaction_frame.index.get_level_values("obs_id")
alt_index = interaction_frame.index.get_level_values("alt_id")

interactions = {}

# income_x_rent interaction (only if household_income is available)
if "household_income" in choosers.columns:
    household_income = pd.to_numeric(choosers["household_income"], errors="coerce").fillna(0.0)
    interaction_frame["income_x_rent"] = (
        household_income.reindex(obs_index).to_numpy()
        * alternatives["median_contract_rent"].reindex(alt_index).to_numpy()
    )
    interactions["income_x_rent"] = interaction_frame["income_x_rent"]

# owner_x_home_value interaction (only if tenure is available)
if "tenure" in choosers.columns:
    owner_flag = (
        choosers["tenure"].astype(str).str.contains("owner", case=False, na=False).astype(float)
    )
    interaction_frame["owner_x_home_value"] = (
        owner_flag.reindex(obs_index).to_numpy()
        * alternatives["median_home_value"].reindex(alt_index).to_numpy()
    )
    interactions["owner_x_home_value"] = interaction_frame["owner_x_home_value"]

interactions

{'owner_x_home_value': obs_id  alt_id     
 0       06111001301   -0.0
         06111003011   -0.0
         06111007201    0.0
         06111007402    0.0
         06111007507    0.0
                       ... 
 42754   06111004704   -0.0
         06111004710   -0.0
         06111005504    0.0
         06111007511    0.0
         06111008600   -0.0
 Name: owner_x_home_value, Length: 7439370, dtype: float64}

## Build the locpick ChoiceTable

This uses the full tract choice set. If the study area is large, replace `sample_size=None` with a tract sample size and document the sampling design.


In [10]:
assert choosers.index.is_unique
assert alternatives.index.is_unique
assert choosers["chosen_tract"].isin(alternatives.index).all()

ct = ChoiceTable.from_tables(
    choosers=choosers,
    alternatives=alternatives,
    chosen_alternatives="chosen_tract",
    matrix_data=interactions,
    sample_size=None,
    seed=RANDOM_STATE,
)

ct

ChoiceTable(n_obs=42755, n_alts=174, choice_col='chosen')

## Estimate Baseline and Interaction Models

Start with a simple baseline, then add the chooser-by-tract interaction terms. Depending on how `household_income` is encoded by `livelike`, you may want to recode the chooser-side categorical variables before relying on coefficient magnitudes.


In [11]:
formula_1 = "median_contract_rent + median_home_value + median_household_income + p_poverty_rate"

model_1 = ChoiceModel(ct, formula=formula_1)
result_1 = model_1.fit()

In [12]:
print(result_1.summary())

Multinomial Logit Estimation Results
Observations:           42755
Alternatives:             174
Parameters:                4
DF residual:            42751
------------------------------------------------------------
Log-likelihood: -213419.3141
LL (null):       -220575.4093
AIC:              426846.6282
BIC:              426881.2811
Rho-squared:           0.0324
Adj. rho-sq:           0.0324
------------------------------------------------------------
Parameter                  Coef    Std.Err        t    P>|t|
------------------------------------------------------------
median_contract_rent     0.0539     0.0056    9.666   0.0000
median_home_value       -0.6571     0.0103  -63.979   0.0000
median_household_income     0.4965     0.0110   45.222   0.0000
p_poverty_rate          -0.6209     0.0106  -58.321   0.0000


In [13]:
# Build formula dynamically based on which interactions are present in the ChoiceTable
_base_terms = [
    "median_contract_rent",
    "median_home_value",
    "median_household_income",
    "p_poverty_rate",
]
_interaction_terms = [k for k in interactions.keys() if k in ct._ds.data_vars]
formula_2 = " + ".join(_base_terms + _interaction_terms)

model_2 = ChoiceModel(ct, formula=formula_2)
result_2 = model_2.fit()
print(result_2.summary())

Multinomial Logit Estimation Results
Observations:           42755
Alternatives:             174
Parameters:                4
DF residual:            42751
------------------------------------------------------------
Log-likelihood: -213419.3141
LL (null):       -220575.4093
AIC:              426846.6282
BIC:              426881.2811
Rho-squared:           0.0324
Adj. rho-sq:           0.0324
------------------------------------------------------------
Parameter                  Coef    Std.Err        t    P>|t|
------------------------------------------------------------
median_contract_rent     0.0539     0.0056    9.666   0.0000
median_home_value       -0.6571     0.0103  -63.979   0.0000
median_household_income     0.4965     0.0110   45.222   0.0000
p_poverty_rate          -0.6209     0.0106  -58.321   0.0000


In [14]:
result_1.to_latex().encode("unicode_escape").decode().replace("\\\\", "\\")

'\\begin{tabular}{lcccc}\\n\\hline\\nParameter & Coef & Std.Err & t & P>|t| \\\\\\n\\hline\\nmedian_contract_rent & 0.0539 & 0.0056 & 9.666 & 0.0000 \\\\\\nmedian_home_value & -0.6571 & 0.0103 & -63.979 & 0.0000 \\\\\\nmedian_household_income & 0.4965 & 0.0110 & 45.222 & 0.0000 \\\\\\np_poverty_rate & -0.6209 & 0.0106 & -58.321 & 0.0000 \\\\\\n\\hline\\n\\end{tabular}'

In [15]:
from IPython.display import Math

display(Math(result_2.to_latex().encode("unicode_escape").decode().replace("\\\\", "\\")))

<IPython.core.display.Math object>

In [16]:
pd.DataFrame(model_2.probabilities())

,0,1,2,3,4,5,6,7,8,9,...,164,165,166,167,168,169,170,171,172,173
0,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
1,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
2,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
3,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
4,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42750,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
42751,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
42752,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889
42753,0.00597,0.002797,0.00981,0.006784,0.017542,0.006261,0.009191,0.007667,0.00739,0.005115,...,0.006744,0.001107,0.003988,0.006459,0.005343,0.006448,0.004552,0.007778,0.00185,0.001889


In [17]:
model_2.average_marginal_effect("median_contract_rent").mean()

np.float64(0.053566054881651184)

## Interpretation Caveat

This notebook demonstrates how to build and estimate a tract choice model on a synthetic household population. The estimated coefficients summarize relationships in the synthetic tract allocations produced by the `livelike` workflow.

They should not be interpreted as validated behavioral parameters for observed residential choice without additional calibration or external observed-choice data.
